In [12]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [13]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, 10)
        # self.fc1 = nn.Linear(hidden_size, 10)
        # self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc0(x)
        return x

In [14]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()), # replace to default
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.4},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    # "min_epochs_between_expansions": 10_000,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-4,
    "diff_threshold": 0.05
}

In [15]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [16]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

Converting layer 'fc0' to ScaledNet...


In [17]:
# import wandb
# wandb.login()

In [18]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [19]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key="r0A4z5VJhRpfdOKvs2kuqouv3",
    workspace="shi-van",
    project_name="split_data",
)
experiment.log_parameters(hyperparams)
experiment.set_name("fashion_mnist origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/shi-van/split-data/859870a291ac4932b88c682ca3c40a57



In [20]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(m):
    return optim.Adam(m.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])

trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True,
)

--- Starting SSONN Training ---


Epoch 1/256 | T-Loss: 0.8992 | V-Loss: 0.6473 | Met: 0.7865 | Params: 7850/7850


Epoch 2/256 | T-Loss: 0.5887 | V-Loss: 0.5555 | Met: 0.8123 | Params: 7850/7850


Epoch 3/256 | T-Loss: 0.5288 | V-Loss: 0.5195 | Met: 0.8245 | Params: 7850/7850


Epoch 4/256 | T-Loss: 0.4965 | V-Loss: 0.4940 | Met: 0.8307 | Params: 7850/7850


Epoch 5/256 | T-Loss: 0.4766 | V-Loss: 0.4803 | Met: 0.8364 | Params: 7850/7850


... (102 outputs left out)

Epoch 107/256 | T-Loss: 0.1399 | V-Loss: 0.3908 | Met: 0.8798 | Params: 1838849/1859779


Epoch 108/256 | T-Loss: 0.1384 | V-Loss: 0.4082 | Met: 0.8843 | Params: 1838849/1859779


DEBUG: Epoch 108, Avg Change: 0.00294, Threshold: 0.05
   -> Plateau detected (change 0.00294).
   -> Action: Expanding Model.
[Expansion] Layer 'fc0': Adding sub-layer for 5 edges.
   -> Re-initializing optimizer and resetting LR.
Epoch 109/256 | T-Loss: 0.1378 | V-Loss: 0.3956 | Met: 0.8824 | Params: 1853234/1874219


KeyboardInterrupt: 

In [21]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : fashion_mnist origin
COMET INFO:     url                   : https://www.comet.com/shi-van/split-data/859870a291ac4932b88c682ca3c40a57
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     expansion event [7] : (5, 2000)
COMET INFO:     lr [109]            : (0.00025, 0.001)
COMET INFO:     params amount [109] : (7850, 1853234)
COMET INFO:     sparsity [109]      : (0.0, 0.01376850142378827)
COMET INFO:     total_params [109]  : (7850, 1874219)
COMET INFO:     train loss [109]    : (0.12579396759417463, 0.8992445129663387)
COMET INFO:     val accuracy [109]  : (0.7865, 0.89425)
COMET INFO:     val loss [109]      : (0.32020267273517367, 0.6472594

In [ ]:
torch.save(model, 'fashion_mnist.pt')